# Preparation

- Append galaxies in subfields to HSC fileds

- Mask data

```python
"""
# sim_analysis_utilities.prepare_catalog
# Read the catalog data
"""
catalog =   Table.read(filename)
colNames=   catalog.colnames

# Load the header to get proper name of flags
header = pyfits.getheader(filename, 1)
n_flag = catalog["flags"].shape[1]
for i in range(n_flag):
    catalog[header["TFLAG%s" % (i+1)]] = catalog["flags"][:, i]

# Then, apply mask for permissive cuts
mask =  (~(catalog['base_SdssCentroid_flag'])) &\
        (~catalog['ext_shapeHSM_HsmShapeRegauss_flag']) &\
        (catalog['base_ClassificationExtendedness_value'] > 0) &\
        (~np.isnan(catalog['modelfit_CModel_flux']))&\
        (~np.isnan(catalog['modelfit_CModel_fluxSigma'])) &\
        (~np.isnan(catalog['ext_shapeHSM_HsmShapeRegauss_resolution'])) &\
        (~np.isnan(catalog['ext_shapeHSM_HsmPsfMoments_xx'])) &\
        (~np.isnan(catalog['ext_shapeHSM_HsmPsfMoments_yy'])) &\
        (~np.isnan(catalog['ext_shapeHSM_HsmPsfMoments_xy'])) &\
        (~np.isnan(catalog['base_Variance_value']))&\
        (~np.isnan(catalog['modelfit_CModel_flux']))&\
        (~np.isnan(catalog['modelfit_CModel_fluxSigma']))&\
        (~np.isnan(catalog['ext_shapeHSM_HsmShapeRegauss_resolution']))&\
        (catalog['deblend_nChild']  ==  0)
catalog =   catalog[mask]
if len(catalog)==0:
    return None
catalog =   catalog[colNames]

# Add columns with names matching the real data
cols = []
data_fields = {'ext_shapeHSM_HsmShapeRegauss_e1': 'i_hsmshaperegauss_e1',
               'ext_shapeHSM_HsmShapeRegauss_e2': 'i_hsmshaperegauss_e2',
               'ext_shapeHSM_HsmShapeRegauss_sigma': 'i_hsmshaperegauss_sigma',
               'ext_shapeHSM_HsmShapeRegauss_resolution': 'i_hsmshaperegauss_resolution'}
for i in data_fields:
    catalog[data_fields[i]] = catalog[i]

# Add field name infomation
catalog['field_name']   = np.string_(fieldname)
if isim>=0:
    """
    # For simulation
    """
    catalog['isim'] =   isim
    catalog['paired']= False
    #Reweight factor to be determined
    catalog['reweight_s16a']=  0.

    catalog['centDist'] =   ((catalog['base_SdssCentroid_y']%ngrid-ngrid/2)**2. +
                            (catalog['base_SdssCentroid_x']%ngrid-ngrid/2)**2.)
    catalog['centDist'] =   np.sqrt(catalog['centDist'])
    catalog             =   catalog[(catalog['centDist']<5.)]

    # First, keep only detections that are the closest to the grid point
    # Get sorted index by grid index and grid distance
    # Add a gridIndex column instead of using the index column, for compatibility
    catalog['ipos']     =   (catalog['base_SdssCentroid_y']//ngrid)*ngalR +(catalog['base_SdssCentroid_x']//ngrid)
    catalog['ipos']     =   catalog['ipos'].astype(np.int)
    inds        =   np.lexsort([catalog['centDist'], catalog['ipos']])
    catalog     =   catalog[inds]
    inds_unique =   np.unique(catalog['ipos'], return_index=True)[1]

    catU    =   catalog[inds_unique]
    """
    # Identify pairs based on their input ipos values
    """
    n   = ngalR*ngalR# Number of grid points
    mask_even = np.in1d(catU['ipos'] % n, 2*np.arange(n/2).astype('int'), assume_unique=True)
    mask_odd  = np.in1d(catU['ipos'] % n, 2*np.arange(n/2).astype('int')+1, assume_unique=True)

    # Find index of broken pairs
    broken_pairs = np.setxor1d(catU['ipos'][mask_even] % n,
            catU['ipos'][mask_odd] % n - 1)
    mask_pairs = ~np.in1d(catU['ipos'] % n,
            np.concatenate([broken_pairs, broken_pairs+1]), assume_unique=True)

    catalog['paired'][inds_unique] = mask_pairs
    print(sum(catalog['paired']))
    catalog.sort(['paired', 'ipos'])

    """
    # Load information from psfPre
    """
    psfFname    =   'psfPre/psf-%05d.fits' % isim
    psfInfo     =   pyfits.getheader(psfFname)
    catalog['g1']=  psfInfo['g1']
    catalog['g2']=  psfInfo['g2']

    """
    # Load information from catPre
    """
    catFname    =   'catPre/catalog-%05d.fits' % isim
    catInfo     =   Table.read(catFname)
    # Join catalogs on grid index
    # Extinction information is contained in catInfo
    catalog = join(catalog, catInfo, 'ipos')

# Add column to make a WL flag for the simulation data:
#     imag_cmodel - a_i < 24.5
#     sigma_e > 0, < 0.4
#     |e|<2
#     resolution factor
#     i-band SNR
#     blendedness cut
sig_e   =   get_sigma_e(catalog)
e       =   np.sqrt(catalog['i_hsmshaperegauss_e1']**2 +
           catalog['i_hsmshaperegauss_e2']**2)

catalog['weak_lensing_flag'] = ((get_imag(catalog)-catalog['a_i']) < 24.5) & \
    (e <= 2.) & \
    (catalog['i_hsmshaperegauss_resolution'] >= 0.3) & \
    (get_snr(catalog) >= 10.) & \
    (sig_e > 0.0) & \
    (sig_e < 0.4) & \
    (catalog['base_Blendedness_abs_flux']<= 0.4217)

# add a bunch of columns that get used for WL analysis - fake it for now!
catalog['i_hsmshaperegauss_derived_sigma_e'] = catalog['i_hsmshaperegauss_sigma']
catalog['i_hsmshaperegauss_derived_rms_e'] = 0.365

wt =  1./(catalog['i_hsmshaperegauss_derived_rms_e']**2 + \
             catalog['i_hsmshaperegauss_derived_sigma_e']**2)
catalog['i_hsmshaperegauss_derived_shape_weight'] = wt
# Unpleasant discovery: the lines below used to say "= 0" and it turns out that meant those
# columns were ints, which led to all kinds of issues later on.  This fix is for the next time a
# sims catalog gets turned into HDF5.
catalog['i_hsmshaperegauss_derived_shear_bias_m'] = 0.
catalog['i_hsmshaperegauss_derived_shear_bias_c1'] = 0.
catalog['i_hsmshaperegauss_derived_shear_bias_c2'] = 0.
```

# Optimal Weights

## $\sigma_e$

## $e_{\rm{rms}}$